# 15 | Nested Training-Data Scaling: Random Forest and Weighted CNN

**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**Aim:** Compare 500 versus 1,000 maximum training windows per participant while holding **the same six development Validation people and their exact original 500-cap windows** fixed. The 1,000-cap Train includes every original 500-cap Train window. Protected original Test IDs S023/S032 are never loaded. This is a development ablation, not independent validation.

## 01. Imports and reusable methods

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score

# Find the repository without introducing a custom installable package.
root = Path.cwd().resolve()
while not (root / "code/python/src/core/data_loading.py").is_file():
    if root.parent == root: raise RuntimeError("Open inside the repository.")
    root = root.parent
sys.path.insert(0, str(root / "code/python/src"))
sys.path.insert(0, str(root / "studies/wearable_activity_recognition/src"))
from harth_data import HARTH_SUBJECTS, SENSOR_COLUMNS, ACTIVITY_NAMES, prepare_harth_nested_window_cohort
from core.data_splitting import split_group_indices
from core.feature_engineering import extract_signal_features
from core.temporal_cnn import train_temporal_cnn
from core.ml_baselines import classification_class_report, evaluate_predictions_by_subject
from core.group_cross_validation import compare_models_by_subject
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 02. Build source-aligned nested windows for development participants

All recordings are segmented individually before excluding undocumented and out-of-task labels. A participant's 500-cap windows are **strictly contained** in their expanded 1,000-cap set. Only previously unused eligible positions are added.

In [ ]:
RESERVED_TEST = {"S023", "S032"}
DEVELOPMENT_SUBJECTS = sorted(set(HARTH_SUBJECTS) - RESERVED_TEST)
TASK_CLASSES = tuple(range(1, 9))
X, metadata, overview = prepare_harth_nested_window_cohort(
    subjects=DEVELOPMENT_SUBJECTS,
    data_dir=root / "data/raw/harth_v2",
    task_classes=TASK_CLASSES,
    base_cap=500, expanded_cap=1000,
    window_size=100, step_size=50, sampling_rate=50,
)
assert metadata["subject_id"].nunique() == 29
assert not (set(metadata["subject_id"]) & RESERVED_TEST)
assert overview["base_windows"].sum() == 14201
print("Expanded source window array:", X.shape)
display(overview)

## 03. Freeze the original Stage 10–12 subject split and Validation windows

The split is reproduced using the original **base 500-window cohort**, not an independently resampled split. The three unused development subjects remain excluded from model fitting.

In [ ]:
# Recreate the exact Stage 10 group allocation using original base rows.
base_mask = metadata["in_base_500"].to_numpy(dtype=bool)
base_row_ids = np.flatnonzero(base_mask)
original_parts = split_group_indices(
    metadata.iloc[base_row_ids]["subject_id"].to_numpy(),
    train_fraction=0.70, validation_fraction=0.20,
    test_fraction=0.10, random_state=42,
)
train_people = set(metadata.iloc[base_row_ids[original_parts["train"]]]["subject_id"])
validation_people = set(metadata.iloc[base_row_ids[original_parts["validation"]]]["subject_id"])
unused_people = set(metadata.iloc[base_row_ids[original_parts["test"]]]["subject_id"])
assert len(train_people) == 20 and len(validation_people) == 6 and len(unused_people) == 3
assert not (train_people & validation_people or train_people & unused_people or validation_people & unused_people)

person_ids = metadata["subject_id"].to_numpy()
validation_rows = np.flatnonzero(base_mask & np.isin(person_ids, list(validation_people)))
base_train_rows = np.flatnonzero(base_mask & np.isin(person_ids, list(train_people)))
expanded_train_rows = np.flatnonzero(np.isin(person_ids, list(train_people)))
assert np.isin(base_train_rows, expanded_train_rows).all()
assert len(validation_rows) == 2701
print("Train people:", sorted(train_people))
print("Validation people:", sorted(validation_people))
print("Train windows (base/expanded):", len(base_train_rows), len(expanded_train_rows))
print("Validation windows (fixed):", len(validation_rows))

## 04. Audit the exact training-class increase

Report what is added by larger Train sets. The Validation class labels and rows remain fixed in both conditions.

In [ ]:
# Class counts before model fitting.
train_class_counts = pd.DataFrame({
    "base_train": metadata.iloc[base_train_rows]["label"].value_counts(),
    "expanded_train": metadata.iloc[expanded_train_rows]["label"].value_counts(),
    "fixed_validation": metadata.iloc[validation_rows]["label"].value_counts(),
}).reindex(TASK_CLASSES).fillna(0).astype(int)
train_class_counts.index = [f"{code}: {ACTIVITY_NAMES[code]}" for code in TASK_CLASSES]
train_class_counts["new_train_windows"] = (
    train_class_counts["expanded_train"] - train_class_counts["base_train"]
)
display(train_class_counts)
assert (train_class_counts["base_train"] > 0).all()
assert (train_class_counts["new_train_windows"] >= 0).all()

## 05. Fit Random Forest on both training-data sizes

Extract the same five deterministic statistical features per sensor. Both models use the same RF hyperparameters and exactly the same Validation windows.

In [ ]:
# Extract features once from the shared actual source windows.
features = extract_signal_features(X, channel_names=SENSOR_COLUMNS)
y = metadata["label"].to_numpy()
rf_predictions = {}
for name, train_rows in [("Train 500", base_train_rows), ("Train 1000", expanded_train_rows)]:
    forest = RandomForestClassifier(
        n_estimators=120, min_samples_leaf=3,
        class_weight="balanced_subsample", random_state=42, n_jobs=-1
    )
    forest.fit(features.iloc[train_rows], y[train_rows])
    rf_predictions[name] = forest.predict(features.iloc[validation_rows])
    print(name, "RF Validation Macro F1:",
          round(f1_score(y[validation_rows], rf_predictions[name],
                         labels=TASK_CLASSES, average="macro", zero_division=0), 4))

## 06. Fit Weighted CNN on both nested training sizes

Train-only channel standardisation and train-only inverse-square-root loss weights are independently refitted for each training-data size. Architecture, seed, epoch budget and fixed Validation set remain the same.

In [ ]:
# The same bounded architecture is trained on two strictly nested Train sets.
cnn_predictions, histories = {}, {}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
for name, train_rows in [("Train 500", base_train_rows), ("Train 1000", expanded_train_rows)]:
    print("Weighted CNN:", name)
    model, history, prediction, scaling = train_temporal_cnn(
        X_train=X[train_rows], y_train=y[train_rows],
        X_validation=X[validation_rows], y_validation=y[validation_rows],
        classes=TASK_CLASSES, epochs=5, batch_size=128,
        learning_rate=1e-3, seed=42, device=DEVICE,
        class_weighting="inverse_sqrt",
    )
    cnn_predictions[name] = prediction
    histories[name] = history
    display(history.round(4))

## 07. Compare all four conditions on the same Validation windows

Any change in Validation performance is a development observation associated with more training data and a different optimisation trajectory. It does not establish causal gains or generalisation beyond these six already-inspected participants.

In [ ]:
# Report fixed-task Macro F1 and balanced accuracy on identical source rows.
prediction_sets = {
    "RF 500": rf_predictions["Train 500"],
    "RF 1000": rf_predictions["Train 1000"],
    "Weighted CNN 500": cnn_predictions["Train 500"],
    "Weighted CNN 1000": cnn_predictions["Train 1000"],
}
y_val = y[validation_rows]
results = pd.DataFrame([
    {"model": name, "validation_macro_f1": f1_score(
         y_val, predicted, labels=TASK_CLASSES, average="macro", zero_division=0),
     "balanced_accuracy": balanced_accuracy_score(y_val, predicted),
     "validation_windows": len(y_val)}
    for name, predicted in prediction_sets.items()
])
display(results.round(4))

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(results["model"], results["validation_macro_f1"])
ax.set(title="Training-data scaling on identical development Validation",
       ylabel="Macro F1", ylim=(0, 1))
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()

## 08. Inspect Shuffling, stair activities and participant-level differences

Class support is fixed in Validation. The per-class and per-person comparisons reveal whether increased training data helps difficult activities consistently.

In [ ]:
# Inspect original activity-code precision, recall and F1.
class_tables = []
person_tables = []
val_people_ids = metadata.iloc[validation_rows]["subject_id"].to_numpy()
for name, predicted in prediction_sets.items():
    class_result = classification_class_report(
        y_val, predicted, classes=TASK_CLASSES,
        class_names=[ACTIVITY_NAMES[c] for c in TASK_CLASSES]
    )
    class_result["model"] = name
    class_tables.append(class_result)
    person_tables.append(evaluate_predictions_by_subject(
        y_val, predicted, val_people_ids, TASK_CLASSES, name
    ))
classes_report = pd.concat(class_tables, ignore_index=True)
people_report = pd.concat(person_tables, ignore_index=True)
display(classes_report[["model","class","support","precision","recall","f1"]].round(4))
for larger, smaller in [("RF 1000","RF 500"), ("Weighted CNN 1000","Weighted CNN 500")]:
    paired, summary = compare_models_by_subject(
        people_report, model_a=larger, model_b=smaller
    )
    display(pd.DataFrame([summary]).round(4))
    display(paired.round(4))

## 09. Research quality gate

No original Test subject is loaded. Nested Train sets and identical Validation source-row identities are checked explicitly. Future decisions require wider group-level validation, rather than a conclusion from this already-used development split.

In [ ]:
# Confirm row/participant isolation and fixed Validation integrity.
assert len(base_train_rows) == 10000
assert len(validation_rows) == 2701
assert np.isin(base_train_rows, expanded_train_rows).all()
assert not (set(val_people_ids) & train_people)
assert not (set(val_people_ids) & RESERVED_TEST)
assert all(len(predicted) == len(y_val) for predicted in prediction_sets.values())
assert results["validation_macro_f1"].between(0, 1).all()
print("Nested training-data size ablation: PASSED")
print("Validation identities and original Test protection: PASSED")
print("Independent full-dataset generalisation: NOT ESTABLISHED")